# Пояснения к работе с API и Логгированию

В этом разделе, я делально рабрезу, как я фетчил API: скажу, почему я принимал те или иные решения в процессе разработки, проясню технические нюансы

## Выбор инструментов и их импорт

В процессе работы с API, нам потребовались более продвинутые инструменты, так как суммарно, мы сделали около *500 000* запросов, и, естественно, возникли проблемы с rate limits, а также обработкой ошибок - когда мы делаем такое большое количество запросов, нам крайне важно минимизировать неудавшиеся фетчи, и одновременно обезопасить себе, если они все таки произошли.

### Наш подход

#### Rate-limits

Для того, чтобы решить проблему с rate-limits - мы решили использовать *100* прокси, и выполнять запросы асинхронно.

Библиотека [***Requests***](https://requests.readthedocs.io/en/latest/) не поддерживает этот функционал, поэтому пришлось подключать другую. Наш выбор пал на пакет [***HTTPX***](https://www.python-httpx.org/). Это достаточно свежий и удобный инструмент, который позволяет легко выполнять асинхронные запросы, по сигнатурам, похожие на обычные из [***Requests***](https://requests.readthedocs.io/en/latest/).

Прокси я приобрел на сайте [***Webshare.IO***](https://www.webshare.io/), и фетчу полный список по его внутреннему API, а позже обрабатываю, приводя к нужному формату. Это все выполняется в файле ```config.py```

#### Обработка ошибок и логгирование

Для обработки ошибок в критических местах (массовые запросы по **API**) мы использовали стандартные ```try/expect```, 
а также библиотеку [***tenacity***](https://tenacity.readthedocs.io/en/latest/), для автоматического повторения проблемных запросов, через определенный промежуток времени (через декоратор).

Для логгирования, была выбрана библиотека [***loguru***](https://loguru.readthedocs.io/en/stable/). Это обертка для стандартным логгером Python, которая упрощает работу с ним. Она уже имеет несколько реализованных уровней логгирования, грамотный **trace-back**, а также готовый декоратор, для автоматического логгирования любых ошибок в функциях.


Ну что же, приступим к импорту этого добра.





In [ ]:
import random
import requests
import json
from copy import deepcopy
import pandas as pd
from loguru import logger
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential
import config
import httpx
import asyncio
import numpy as np

logger.add(".log/{time}.log")

## Основная часть 

Теперь можно перейти непосредственно к работе с API.

Первым делом, следует сфетчить самую главную информацию - ***id*** всех игр со стима.

### Appids всех игр

Разобьем этот процесс на две функции: 

- ```get_next_dataset``` - возвращает ответ в нужном формате
- ```get_all_steam_appids``` - возвращает ***appid*** всех игр

In [ ]:
@logger.catch
def get_next_dataset(params):
  return requests.get(config.all_games_url, params).json()["response"]
  


@logger.catch
def get_full_appids():
  
  last = get_next_dataset(config.data_private_api_params)
  acc = deepcopy(last["apps"])

  while "have_more_results" in last.keys():
    logger.success(f"Steam API dataset successfully fetched last game id: {last['last_appid']}")

    last = get_next_dataset({**config.data_private_api_params, "last_appid": last["last_appid"]})

    acc += last["apps"]
    
  logger.success(f"Fetching ended successfully. Total lines: {len(acc)}")  

  return pd.DataFrame(acc)

Как вы видите, запрос на получение игр у нас повторяется в цикле, до тех пор, пока ```have_more_results``` возвращается нам как одно из полей ответа.

Это нужно, потому что за один запрос можно сфетчить максимум *50 000* айдишников игр. Каждую итерацию мы запоминаем id последней игры, которая была возвращена, и передаем его как параметр начала при дальнейшем запросе.

Таким образом, после завершения функции мы получаем список ***appid*** (и еще некоторую другую информацию) вообще обо всех играх которые есть в Steam.


Вы можете заметить, что обе функции используют декоратор ```@logger.catch```. Я про него уже говорил выше, он автоматические логгирует все ошибки которые происходят внутри функции с грамотным ***trace-back***.




### Фетчинг детальной информации о каждой игре по ее ***appid***

Перед тем как начать непосредственно фетчить, нам нужно разобраться как делать это асинхронно, через большое количество прокси.


Схема такая: 

Создаем много асинхронных клиентов библиотеки ***HTTPX*** со своим прокси, делим ***appid*** всех игр на равные группы и запускаем асинхронно все клиенты, чтобы каждый из них ПОСЛЕДОВАТЕЛЬНО фетчил свою группу с определенной задержкой перед запросами (для того, чтобы не получить rate-limit).

Далее всю собранную информацию мы агрегируем в один датасет

Начнем с разбора функции работы отдельного клиента.


In [ ]:
@retry(wait=wait_exponential(max=60, min=5), stop=stop_after_attempt(3), retry=retry_if_exception_type(httpx.ReadTimeout))
async def get_with_retry(client, url):
  return await client.get(url)

@logger.catch
async def fetch_steam_api(client, urls, ids):
  data = dict()  
  await asyncio.sleep(random.uniform(0, 2))

  for i in range(len(urls)):
    try:
      response = await get_with_retry(client, urls[i])
      data[ids[i]] = response.json()
    except Exception as e:
      logger.warning(f"URL {urls[i]} failed to fetch\n Error: {e}")
    await asyncio.sleep(random.uniform(1, 3))
  logger.success(f"client {id(client)} successfully fetched {urls[0]} group")
  return data  



Функция ```fetch_steam_api``` делает ровно то, что я описал выше. 

Стоит только дополнительно уточнить, что мы ловим ошибку и логгируем ее, если запрос неудачный, а так же НЕ добавляем response этого запроса в датасет.

Также для безопасности используем элемент рандома при задержках между запросами (чтобы реквесты выглядели более "натурально") и между запускам каждого клиента мы также создаем рандомную задержку, чтобы все клиенты не долбились в эндпоинт синхронно в одно и то же время.

На самом деле это было добавлено, так как множество запросов падало с ошибками, и я решил использовать все методы защиты, которые смог.

Стоит также дополнительно поговорить о функции ```get_with_retry```. 

По сути, это обычный асинхронный **get** запрос, но с прикрученным декоратором ```@retry``` из библиотеки tenacity. 

В нашем случае он позволяет повторять запрос три раза, с экспоненциально возрастающей задержкой между ними, в случае, если запрос вернул нам ошибку

Далее введем функцию, которая создает нам корутины и после их выполнения возвращает нам данные

In [ ]:
# urls is a list of url lists generated for each client
@logger.catch
async def fetch_steam_full(clients, urls, ids, list_params=None):

  tasks = [fetch_steam_api(clients[i], urls[i], ids[i]) for i in range(len(urls))] 
  
  data = await asyncio.gather(*tasks)

  return data


Ну и наконец перейдем входной функции для всего процесса фетчинга детальной информации об играх.

В этой функции мы:
1. Создаем клиенты с индивидуальным прокси
2. Разделяем массив appids на массив массивов, каждый элемент в котором, это список appid который будет использовать каждый отдельный клиент
3. Создаем подобный массив для эндпоинтов (нужно форматирование, так как часть appid передается не как параметр а как часть пути ```/appid```)
4. Ну и запускаем функцию сбора данных для всех клиентов.

Стоит отдельно отметить, что цикл ```url in urls``` нам нужен, так как подобным методом мы фетчим несколько эндпоинтов. В конце возвращаем список уже сагрегированных данных для каждого из них.


In [ ]:
@logger.catch
async def fetch_all_pub_steam_sources(appids, urls):
  clients = list(map(lambda p: httpx.AsyncClient(proxy=p, timeout=20), config.PROXIES))
  logger.success(f"{len(clients)} clients created")
  splitted_ids = [i.astype(int).tolist() for i in np.array_split(appids, len(config.PROXIES))]

  data = list()

  for url in urls:
    
    url_list = list(map(lambda y: list(map(lambda x: url.format(appid=x), y)), splitted_ids))
    current_data = await fetch_steam_full(clients, url_list, splitted_ids)
    data.append(current_data)
    logger.success(f"Data for url: {url} successfully fetched")


  logger.success("List of all data returning...")
  return data

### Фетчинг дополнительных приватных эндпоинтов ***Steam API***

Тут мы собираем все данные по дополнительным приватным API. Список этих API и соответствующих параметров, находится в ```config.py```.

Запросов не слишком много, и проблем с ними возникнуть не должно, так что мы не будем использовать продвинутые функции выше с созданием отдельных URL, разбиением и тд. Во первых, это оверкилл, а во вторых, для унификации пришлось бы менять эти функции.

Просто создадим асинхронный прокси клиент для каждой url. В крайнем случае, ```@logger.catch``` все равно словит ошибку и мы сможем подумать что можно сделать дополнительно.

In [ ]:
@logger.catch
async def get_all_private_steam_once(urls, params):
  clients = list(map(lambda p: httpx.AsyncClient(proxy=p, timeout=20), config.PROXIES[:len(urls) + 1]))
  tasks = [client.get(url, params=param) for client, url, param in zip(clients, urls, params)]

  data = await asyncio.gather(*tasks)
  return data

Комментировать особо нечего, просто создали клиентов для каждой URL с прокси, создали такси и асинхронно выполнили корутины.

Переходим с фетчингу данных, связанных с конкретным временем. 

Мы говорим о топах, за конкретный месяц/год.


Вот тут мы уже использовали функцию ```fetch_steam_full``` так как ожидается много похожих запросов.

In [ ]:
from datetime import datetime

@logger.catch
async def get_all_top_games_timed():

  months = []
  years = []
  
  for year in range(2003, 2027):
    years.append(datetime(year, 1, 1).timestamp())
    for month in range(1, 13):
      months.append(datetime(year, month, 1).timestamp())

  month_urls = [config.month_top_url.format(key=os.environ["API_KEY"], time=month) for month in months]
  year_urls = [config.year_top_url.format(key=os.environ["API_KEY"], time=year) for year in years]
  
  splitted_month_urls = [i.tolist() for i in np.array_split(month_urls, min(len(config.PROXIES), len(month_urls)) )]
  splitted_year_urls = [i.tolist() for i in np.array_split(year_urls, min(len(config.PROXIES), len(year_urls)))]

  month_ids = [i.astype(int).tolist() for i in np.array_split([int(month) for month in months], len(config.PROXIES))]
  year_ids = [i.astype(int).tolist() for i in np.array_split([int(year) for year in months], len(config.PROXIES))]
  
  clients = list(map(lambda p: httpx.AsyncClient(proxy=p, timeout=20), config.PROXIES))
  month_data = await fetch_steam_full(clients, splitted_month_urls, month_ids)
  logger.success("Top month game data fetched successfully fetched")
  year_data = await fetch_steam_full(clients, splitted_year_urls, year_ids)
  logger.success("Top year game data fetched successfully fetched")

  return (month_data, year_data)

Опишем детальнее как работает этот метод:

1. Так как мы получаем информацию за конкретный период, энпоинты требуют timestamp соответствующий конкретной дате - перед запросами, нам надо создать эти timestamps. Сохраняем их в массивы ```months``` и ```years```
2. Теперь с помощью ```format``` сгенерируем URL для каждого запроса с соответствующими параметрами
3. Как мы помним, ```fetch_steam_full``` принимает, разбитый массив массивов, так что разобьем сгенерированные URL. обратите внимание, что, например, различных годов у нас меньше, чем прокси, так что берем минимум между количеством прокси и количеством элементов в массивах months/years
4. Теперь разбитые массивы для timestamps. Они будут служить ключами, в финальных данных (аналог ***appid*** в контексте наших временных данных)


Дальше просто создаем клиенты и вызываем функцию ```fetch_steam_full``` отдельно для месяцев и годов.

### Заключительный этап

Ну вот и все, вся логика готова и объяснена. 

Осталось только запустить все функции, передавая данные между ними и сохранить результаты в json (так как данные у нас в виде словарей, обрабатывать их будем отдельно).

Скажу только дополнительно, что ```asyncio.run``` нужен, чтобы запускать асинхронные корутины вне асинхронных функций.

In [ ]:
ids = get_full_appids()

with open("game_ids_names.json", "w") as f:
  ids.to_json(f)


data_tuple = asyncio.run(fetch_all_pub_steam_sources(ids, (
  config.app_details_url,
  config.reviews_url,
  config.current_online_url
)))


with open("details_data2.json", "w") as f:
    json.dump(data_tuple[0], f)

with open("reviews_data2.json", "w") as f:
    json.dump(data_tuple[1], f)

with open("current_online_data2.json", "w") as f:
    json.dump(data_tuple[2], f)
    

data = asyncio.run(get_all_private_steam_once(
    [i["url"] for i in config.private_urls_list.values()],
    [i["params"] for i in config.private_urls_list.values()]
))

for dataset, name in zip(data, config.private_urls_list.keys()):
    with open(f"data/{name}.json", "w") as f:
        json.dump(dataset.json()["response"], f)
        

time_data_tuple = asyncio.run(get_all_top_games_timed())
with open("data/month_top_games_data.json", "w") as f:
    json.dump(time_data_tuple[0], f)

with open("data/year_top_games_data.json", "w") as f:
    json.dump(time_data_tuple[1], f)